In [15]:
#!/usr/bin/env python
# coding: utf-8

# In[32]:
import torch
import torch.nn as nn
import ltn
import numpy as np
import matplotlib.pyplot as plt

In [16]:
# In[33]:
# Configure o LTNtorch para usar o dispositivo apropriado.
ltn.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {ltn.device}")

Using device: cpu


In [17]:
# In[34]:
# 1. DEFINIR CONECTIVOS LÓGICOS E QUANTIFICADORES (REQUERIDOS NO LTNTORCH)
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
sat_agg = ltn.fuzzy_ops.SatAgg()

In [18]:
# In[35]:
# 2. CRIAR OBJETOS LTN (Constantes e Variáveis)
red_ball = ltn.Constant(torch.tensor([1.0, 0.0, 0.0]), trainable=False)
blue_cube = ltn.Constant(torch.tensor([0.0, 0.0, 1.0]), trainable=False)

balls = ltn.Variable("balls", torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.5, 0.0, 0.5],
]))
cubes = ltn.Variable("cubes", torch.tensor([
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 0.0],
    [0.5, 0.5, 0.5],
]))

In [19]:
# In[36]:
# 3. DEFINIR PREDICADOS (Propriedades/relações lógicas)
class ColorPredicate(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        if hasattr(x, 'value'):
            x = x.value
        return self.network(x).squeeze()

In [20]:
is_red = ltn.Predicate(ColorPredicate())
is_blue = ltn.Predicate(ColorPredicate())
is_round = ltn.Predicate(ColorPredicate())

In [21]:
# 4. DEFINIR AXIOMAS LÓGICOS (Base de conhecimento)
def axioms():
    facts = [
        is_red(red_ball),
        is_blue(blue_cube),
        is_round(red_ball),
        Not(is_round(blue_cube))
    ]
    rules = [
        Forall(balls, is_round(balls)),
        Forall(ltn.Variable("x", torch.randn(10, 3, device=ltn.device)),
            Implies(is_red(ltn.Variable("x", torch.randn(10, 3, device=ltn.device))),
            is_round(ltn.Variable("x", torch.randn(10, 3, device=ltn.device)))))
    ]
    all_axioms = facts + rules
    return sat_agg(*all_axioms)

In [22]:
# 5. VERIFICAÇÃO DE SATISFATÓRIA (Raciocínio)
def check_satisfiability():
    sat_level = axioms()
    print(f"Knowledge base satisfaction level: {sat_level.item():.4f}")
    return sat_level

In [23]:
# 6. TREINAMENTO DOS PREDICADOS (Aprendendo com os dados)
def train_predicates(epochs=100):
    parameters = list(is_red.model.parameters()) + \
                 list(is_blue.model.parameters()) + \
                 list(is_round.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)
    print("\n=== TRAINING PREDICATES ===")
    for epoch in range(epochs):
        optimizer.zero_grad()
        sat_loss = 1 - axioms()
        sat_loss.backward()
        optimizer.step()
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Satisfaction = {1-sat_loss.item():.4f}")

In [26]:
# In[43]:
# 7. QUERYING THE KNOWLEDGE BASE (Inference)
def query_knowledge_base():
    print("\n=== QUERYING KNOWLEDGE BASE ===")
    # Query 1: Is the red ball red?
    query1 = is_red(red_ball)
    print(f"Q1: Is the red ball red? A: {query1.value.item():.4f}")
    
    # Query 2: Is the blue cube round?
    query2 = is_round(blue_cube)
    print(f"Q2: Is the blue cube round? A: {query2.value.item():.4f}")
    
    # Query 3: Are all balls round? (Universal quantification)
    all_balls_round = Forall(balls, is_round(balls))
    print(f"Q3: Are all balls round? A: {all_balls_round.value.item():.4f}")

In [27]:
# 8. EXECUTE O EXEMPLO COMPLETO
print("=== EXEMPLO DE RACIOCÍNIO LÓGICO LTN ===")
initial_sat = check_satisfiability()
train_predicates(epochs=100)
final_sat = check_satisfiability()
query_knowledge_base()
print("\n=== PRINCIPAIS INFORMAÇÕES SOBRE O LTNObjects ===")
print("1. LTNObjects encapsulam tensores com significado lógico.")
print("2. Constantes representam entidades específicas")
print("3. As variáveis representam conjuntos quantificáveis")
print("4. Os predicados mapeiam LTNObjects para valores de verdade [0,1]")
print("5. Os conectivos lógicos (E, Ou, Não, Implica) operam sobre valores de verdade.")
print("6. Os quantificadores (Forall, Exists) agregam sobre a variávels")
print("7. No LTNTorch, você deve definir explicitamente todos os conectivos lógicos.")
print("8. A função SatAgg combina múltiplos axiomas em uma única pontuação de satisfação.")


=== EXEMPLO DE RACIOCÍNIO LÓGICO LTN ===
Knowledge base satisfaction level: 0.9153

=== TRAINING PREDICATES ===
Epoch 0: Satisfaction = 0.9408
Epoch 20: Satisfaction = 0.9595
Epoch 40: Satisfaction = 0.9154
Epoch 60: Satisfaction = 0.9676
Epoch 80: Satisfaction = 0.9778
Knowledge base satisfaction level: 0.9803

=== QUERYING KNOWLEDGE BASE ===
Q1: Is the red ball red? A: 0.9757
Q2: Is the blue cube round? A: 0.0260
Q3: Are all balls round? A: 0.9898

=== PRINCIPAIS INFORMAÇÕES SOBRE O LTNObjects ===
1. LTNObjects encapsulam tensores com significado lógico.
2. Constantes representam entidades específicas
3. As variáveis representam conjuntos quantificáveis
4. Os predicados mapeiam LTNObjects para valores de verdade [0,1]
5. Os conectivos lógicos (E, Ou, Não, Implica) operam sobre valores de verdade.
6. Os quantificadores (Forall, Exists) agregam sobre a variávels
7. No LTNTorch, você deve definir explicitamente todos os conectivos lógicos.
8. A função SatAgg combina múltiplos axiomas em

In [28]:
# ==============================================================================
# EXTENSÃO: DIFERENTES FORMAS DE IGUALDADE EM LTN (Extensão Original)
# ==============================================================================
# In[45]:
print("\n" + "="*50)
print("EXTENSION: DIFFERENT FORMS OF EQUALITY IN LTN")
print("="*50)


EXTENSION: DIFFERENT FORMS OF EQUALITY IN LTN


In [29]:
Not_eq = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And_eq = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or_eq = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies_eq = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall_eq = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists_eq = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
sat_agg_eq = ltn.fuzzy_ops.SatAgg()

In [35]:
red_object1 = ltn.Constant(torch.tensor([1.0, 0.0, 0.0], device=ltn.device), trainable=True)
red_object2 = ltn.Constant(torch.tensor([0.95, 0.05, 0.0], device=ltn.device), trainable=True)
blue_object = ltn.Constant(torch.tensor([0.0, 0.0, 1.0], device=ltn.device), trainable=True)

objects = ltn.Variable("objetos", torch.tensor([
    [1.0, 0.0, 0.0],
    [0.95, 0.05, 0.0],
    [0.0, 0.0, 1.0],
    [0.0, 1.0, 0.0],
], device=ltn.device))
print("Experimento sobre igualdade: configuração concluída!")

# In[46]: 1. Igualdade diagonal LTN
print("\n" + "-"*30)
print("1. IGUALDADE DIAGONAL LTN")
print("-"*30)

Experimento sobre igualdade: configuração concluída!

------------------------------
1. IGUALDADE DIAGONAL LTN
------------------------------


In [36]:
class DiagonalEquality(nn.Module):
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        cos_sim = torch.nn.functional.cosine_similarity(x_val, y_val, dim=-1)
        return 0.5 * (cos_sim + 1.0)

In [37]:
# Teste da Igualdade diagonal euclidiana
Equal_Diag = ltn.Predicate(DiagonalEquality().to(ltn.device))

In [38]:
def axioms_withEquality_Diag():
    axioms = [
        Forall_eq(objects, Equal_Diag(objects, objects)),
        Equal_Diag(red_object1, red_object2),
        Equal_Diag(red_object2, red_object1),
        Implies_eq(And_eq(Equal_Diag(red_object1, red_object2), Equal_Diag(red_object2, red_object1)), Equal_Diag(red_object1, red_object1)),
        Not_eq(Equal_Diag(red_object1, blue_object))
    ]
    return sat_agg_eq(*axioms)

In [39]:
print("Testando a igualdade diagonal...")
sat_diag = axioms_withEquality_Diag()
print(f"Satisfação KB de igualdade diagonal: {sat_diag.item():.4f}")
print(f"P: Os dois objetos vermelhos são iguais? R: {Equal_Diag(red_object1, red_object2).value.item():.4f}")
print(f"P: Um objeto vermelho é igual a um objeto azul? R: {Equal_Diag(red_object1, blue_object).value.item():.4f}")

# In[50]: 2. Igualdade de cosseno
print("\n" + "-"*30)
print("2. IGUALDADE DE COSSENO.")
print("-"*30)

Testando a igualdade diagonal...
Satisfação KB de igualdade diagonal: 0.7764
P: Os dois objetos vermelhos são iguais? R: 0.9993
P: Um objeto vermelho é igual a um objeto azul? R: 0.5000

------------------------------
2. IGUALDADE DE COSSENO.
------------------------------


In [41]:
class CosineEquality(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.tensor(1.0, device=ltn.device))
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        cos_sim = torch.nn.functional.cosine_similarity(x_val, y_val, dim=-1)
        return torch.sigmoid(self.temperature * (cos_sim + 1.0) / 2.0)
        
Equal_Cos = ltn.Predicate(CosineEquality().to(ltn.device))

In [42]:
def axioms_withEquality_Cos():
    parameters = list(Equal_Cos.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.1)
    print("Treinamento da Igualdade do Cosseno...")
    for epoch in range(200):
        optimizer.zero_grad()
        axioms = [Forall_eq(objects, Equal_Cos(objects, objects)), Equal_Cos(red_object1, red_object2), Equal_Cos(red_object2, red_object1), Not_eq(Equal_Cos(red_object1, blue_object))]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Época {epoch}: Satisfatibilidade da igualdade do cosseno = {sat.item():.4f}, Temp = {Equal_Cos.model.temperature.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Cos(objects, objects)), Equal_Cos(red_object1, red_object2), Not_eq(Equal_Cos(red_object1, blue_object)))

sat_cos = axioms_withEquality_Cos()
print(f"Satisfatibilidade KB da Igualdade do Cosseno Final: {sat_cos.item():.4f}")
print(f"Q: Os dois objetos vermelhos são iguais (cosseno)? A: {Equal_Cos(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (cosseno)? A: {Equal_Cos(red_object1, blue_object).value.item():.4f}")


Treinamento da Igualdade do Cosseno...
Época 0: Satisfatibilidade da igualdade do cosseno = 0.6112, Temp = 1.1000
Época 50: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5738
Época 100: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5680
Época 150: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5687
Satisfatibilidade KB da Igualdade do Cosseno Final: 0.5793
Q: Os dois objetos vermelhos são iguais (cosseno)? A: 0.8275
Q: O objeto vermelho é igual ao objeto azul (cosseno)? A: 0.6866


In [48]:
# In[51]: 3. Igualdade Euclidiana
print("\n" + "-"*30)
print("3. IGUALDADE EUCLIDIANA")
print("-"*30)


------------------------------
3. IGUALDADE EUCLIDIANA
------------------------------


In [49]:
class EuclideanEquality(nn.Module):
    def __init__(self):
        super().__init__()
        self.gamma = nn.Parameter(torch.tensor(0.5, device=ltn.device))
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        distance = torch.sum(torch.square(x_val - y_val), dim=-1)
        return torch.exp(-self.gamma * distance)

Equal_Eucl = ltn.Predicate(EuclideanEquality().to(ltn.device))


In [50]:
def axioms_withEquality_Eucl():
    parameters = list(Equal_Eucl.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.1)
    print("Treinamento em Igualdade Euclidiana...")
    for epoch in range(150):
        optimizer.zero_grad()
        axioms = [Forall_eq(objects, Equal_Eucl(objects, objects)), Equal_Eucl(red_object1, red_object2), Equal_Eucl(red_object2, red_object1), Not_eq(Equal_Eucl(red_object1, blue_object))]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 30 == 0:
            print(f"Época {epoch}: Satisfatibilidade da Igualdade Euclidiana = {sat.item():.4f}, Gamma = {Equal_Eucl.model.gamma.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Eucl(objects, objects)), Equal_Eucl(red_object1, red_object2), Not_eq(Equal_Eucl(red_object1, blue_object)))

sat_eucl = axioms_withEquality_Eucl()
print(f"Satisfatibilidade final da igualdade euclidiana KB: {sat_eucl.item():.4f}")
print(f"Q: Os dois objetos vermelhos são iguais (euclidianos)? A: {Equal_Eucl(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (euclidiano)? A: {Equal_Eucl(red_object1, blue_object).value.item():.4f}")


Treinamento em Igualdade Euclidiana...
Época 0: Satisfatibilidade da Igualdade Euclidiana = 0.8160, Gamma = 0.6000
Época 30: Satisfatibilidade da Igualdade Euclidiana = 0.9903, Gamma = 2.2782
Época 60: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4738
Época 90: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4658
Época 120: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4479
Satisfatibilidade final da igualdade euclidiana KB: 0.9917
Q: Os dois objetos vermelhos são iguais (euclidianos)? A: 0.9879
Q: O objeto vermelho é igual ao objeto azul (euclidiano)? A: 0.0076


In [51]:
# In[52]: 4. Igualdade de Variedades Aprendíveis
print("\n" + "-"*30)
print("4. IGUALDADE DE VARIEDADES APRENDÍVEIS")
print("-"*30)

class LearnableEquality(nn.Module):
    def __init__(self, input_dim=3):
        super(LearnableEquality, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim * 2, 64), nn.ELU(),
            nn.Linear(64, 32), nn.ELU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'valor') else x
        y_val = y.value if hasattr(y, 'valor') else y
        cat_inputs = torch.cat([x_val, y_val], dim=-1)
        return self.net(cat_inputs).squeeze()

Equal_Learned = ltn.Predicate(LearnableEquality(input_dim=3).to(ltn.device))



------------------------------
4. IGUALDADE DE VARIEDADES APRENDÍVEIS
------------------------------


In [52]:
def axioms_withEquality_Learned():
    parameters = list(Equal_Learned.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)
    print("Treinaento da Rede de Igualdade aprodível...")
    for epoch in range(300):
        optimizer.zero_grad()
        axioms = [
            Forall_eq(objects, Equal_Learned(objects, objects)),
            Equal_Learned(red_object1, red_object2), Equal_Learned(red_object2, red_object1),
            Not_eq(Equal_Learned(red_object1, blue_object)), Not_eq(Equal_Learned(blue_object, red_object1)),
            Implies_eq(And_eq(Equal_Learned(red_object1, red_object2), Equal_Learned(red_object2, red_object1)), Equal_Learned(red_object1, red_object1))
        ]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Epoch {epoch}: Satisfatibilidade com a igualdade aprendível= {sat.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Learned(objects, objects)), Equal_Learned(red_object1, red_object2), Not_eq(Equal_Learned(red_object1, blue_object)))

sat_learned = axioms_withEquality_Learned()
print(f"Satisfação final com a base de conhecimento sobre igualdade aprendível: {sat_learned.item():.4f}")
print(f"Q:Os dois objetos vermelhos são iguais (Aprendido)? A: {Equal_Learned(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (aprendido)? A: {Equal_Learned(red_object1, blue_object).value.item():.4f}")
print(f"Q: O objeto vermelho é igual a si mesmo (Aprendido)? A: {Equal_Learned(red_object1, red_object1).value.item():.4f}")


Treinaento da Rede de Igualdade aprodível...
Epoch 0: Satisfatibilidade com a igualdade aprendível= 0.5430
Epoch 50: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 100: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 150: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 200: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 250: Satisfatibilidade com a igualdade aprendível= 0.7959
Satisfação final com a base de conhecimento sobre igualdade aprendível: 0.7113
Q:Os dois objetos vermelhos são iguais (Aprendido)? A: 1.0000
Q: O objeto vermelho é igual ao objeto azul (aprendido)? A: 0.0001
Q: O objeto vermelho é igual a si mesmo (Aprendido)? A: 1.0000


In [ ]:
# ==============================================================================
# NOVA EXTENSÃO: RELAÇÕES GEOMÉTRICAS EM UM PLANO 2D
# ==============================================================================
# In[53]:
print("\n" + "="*60)
print("EXTENSÃO: RELAÇÕES GEOMÉTRICAS EM UM PLANO 2D")
print("="*60)

# Classes (baseadas no centro (x,y) de cada figura):
# - Circle(x), Square(x), Triangle(x)
# Relações espaciais: LeftOf(x,y), Above(x,y), Overlaps(x,y)
#
# AXIOMAS EM LÓGICA DE PRIMEIRA ORDEM (Base de Conhecimento):
# 1. Todos os círculos estão à esquerda de todos os quadrados:
#    ∀x ∀y (Circle(x) ∧ Square(y) → LeftOf(x,y))
# 2. Todos os triângulos estão acima de todos os círculos:
#    ∀x ∀y (Triangle(x) ∧ Circle(y) → Above(x,y))
# 3. A sobreposição é simétrica:
#    ∀x ∀y (Overlaps(x,y) → Overlaps(y,x))
# 4. Nenhum objeto se sobrepõe a si mesmo:
#    ∀x ¬Overlaps(x,x)
#
# AXIOMAS DE ATERRAMENTO
# 5. Rótulos de classe: ∀x∈C Circle(x) ∧ ¬Square(x) ∧ ¬Triangle(x)  (idem para S e T)
# 6. Conversas falsas: ∀s∀c ¬LeftOf(s,c);  ∀c∀t ¬Above(c,t)
# 7. Figuras de classes distintas já relacionadas pelos axiomas 1-2 não se sobrepõem:
#    ∀c∀s ¬Overlaps(c,s);  ∀t∀c ¬Overlaps(t,c)

In [ ]:
# 1. GERAÇÃO DE DADOS (Plano 2D Aleatório)
# As regiões foram escolhidas para serem CONSISTENTES com os axiomas 1 e 2:
#   círculos   : x in [0,4],  y in [0,4]   (canto inferior esquerdo)
#   quadrados  : x in [6,10], y in [0,4]   (canto inferior direito)
#   triângulos : x in [0,10], y in [6,10]  (faixa superior)
torch.manual_seed(42)

def generate_random_shapes(num_per_shape=10):
    dev = ltn.device
    circles   = torch.rand(num_per_shape, 2, device=dev) * torch.tensor([4.0, 4.0], device=dev)
    squares   = torch.rand(num_per_shape, 2, device=dev) * torch.tensor([4.0, 4.0], device=dev) + torch.tensor([6.0, 0.0], device=dev)
    triangles = torch.rand(num_per_shape, 2, device=dev) * torch.tensor([10.0, 4.0], device=dev) + torch.tensor([0.0, 6.0], device=dev)
    return circles, squares, triangles

# CONJUNTO DE TREINO: gerado UMA ÚNICA VEZ (pequeno e restrito: 10 pontos por classe)
train_circles, train_squares, train_triangles = generate_random_shapes(num_per_shape=10)
print("Treino:", train_circles.shape, train_squares.shape, train_triangles.shape)

In [ ]:
# 2. PREDICADOS NEURAIS
SCALE = 10.0  # normaliza coordenadas (0..10) para (0..1): treino mais estável

class ShapeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        x_val = x.value if hasattr(x, 'value') else x
        return self.net(x_val / SCALE).squeeze(-1)

class LearnableSpatialRelation(nn.Module):
    """Predicado binário: extrai UMA feature geométrica do par (x,y) e a passa por uma MLP.
    O LTNtorch já entrega x e y alinhados par a par (um par por linha),
    então NÃO se deve fazer unsqueeze/broadcast aqui dentro."""
    def __init__(self, feature_extractor):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        features = self.feature_extractor(x_val, y_val) / SCALE
        return self.net(features).squeeze(-1)

def left_of_features(x, y):
    return (y[..., 0] - x[..., 0]).unsqueeze(-1)   # positivo se x está à esquerda de y

def above_features(x, y):
    return (x[..., 1] - y[..., 1]).unsqueeze(-1)   # positivo se x está acima de y

def distance_features(x, y):
    return torch.norm(x - y, dim=-1, keepdim=True)  # simétrica por construção

is_circle   = ltn.Predicate(ShapeClassifier().to(ltn.device))
is_square   = ltn.Predicate(ShapeClassifier().to(ltn.device))
is_triangle = ltn.Predicate(ShapeClassifier().to(ltn.device))
left_of  = ltn.Predicate(LearnableSpatialRelation(left_of_features).to(ltn.device))
above    = ltn.Predicate(LearnableSpatialRelation(above_features).to(ltn.device))
overlaps = ltn.Predicate(LearnableSpatialRelation(distance_features).to(ltn.device))

In [ ]:
# 3. AXIOMAS LÓGICOS
def define_geo_axioms(circles, squares, triangles):
    c = ltn.Variable("c", circles)
    s = ltn.Variable("s", squares)
    t = ltn.Variable("t", triangles)

    # --- Axiomas FOL do enunciado (1 a 4) ---
    axioms = [
        # ∀c ∀s (Circle(c) ∧ Square(s) → LeftOf(c,s))
        Forall([c, s], Implies(And(is_circle(c), is_square(s)), left_of(c, s))),
        # ∀t ∀c (Triangle(t) ∧ Circle(c) → Above(t,c))
        Forall([t, c], Implies(And(is_triangle(t), is_circle(c)), above(t, c))),
        # ∀c ∀s (Overlaps(c,s) → Overlaps(s,c))
        Forall([c, s], Implies(overlaps(c, s), overlaps(s, c))),
        # ∀x ¬Overlaps(x,x), para cada classe
        Forall(c, Not(overlaps(c, c))),
        Forall(s, Not(overlaps(s, s))),
        Forall(t, Not(overlaps(t, t))),
        # Desafio extra: ∀x ∀y (Square(x) ∧ Triangle(y) → ¬Overlaps(x,y))
        Forall([s, t], Implies(And(is_square(s), is_triangle(t)), Not(overlaps(s, t)))),
    ]

    # --- Axiomas de ATERRAMENTO (necessários para o treino fazer sentido) ---
    axioms += [
        # Rótulos de classe (sem isso, is_circle = 0 em todo lugar já satisfaria os axiomas 1-2)
        Forall(c, is_circle(c)),   Forall(c, Not(is_square(c))),   Forall(c, Not(is_triangle(c))),
        Forall(s, is_square(s)),   Forall(s, Not(is_circle(s))),   Forall(s, Not(is_triangle(s))),
        Forall(t, is_triangle(t)), Forall(t, Not(is_circle(t))),   Forall(t, Not(is_square(t))),
        # Exemplos negativos das relações (sem isso, LeftOf = 1 para todo par já satisfaria o axioma 1)
        Forall([s, c], Not(left_of(s, c))),
        Forall([c, t], Not(above(c, t))),
        # Círculo/quadrado e triângulo/círculo (pares dos axiomas 1-2) não se sobrepõem
        Forall([c, s], Not(overlaps(c, s))),
        Forall([t, c], Not(overlaps(t, c))),
        Forall([s, t], Not(overlaps(s, t))),
    ]
    return sat_agg(*axioms)

In [ ]:
# 4. TREINAMENTO (somente no conjunto de treino)
def train_geometrical_model(circles, squares, triangles, epochs=500, lr=0.05):
    parameters = []
    for pred in [is_circle, is_square, is_triangle, left_of, above, overlaps]:
        parameters += list(pred.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=lr)
    print("\n=== TREINANDO AXIOMAS GEOMÉTRICOS ===")
    for epoch in range(epochs):
        optimizer.zero_grad()
        sat = define_geo_axioms(circles, squares, triangles)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}: Satisfação = {sat.item():.4f}")

In [ ]:
# 5. TESTE EM NOVOS DADOS ALEATÓRIOS (nunca vistos no treino)
def class_accuracy(pred, own, others):
    """Fração de pontos classificados corretamente por 'pred' (limiar 0.5)."""
    pos = pred(ltn.Variable("x", own)).value > 0.5
    neg = pred(ltn.Variable("x", torch.cat(others))).value <= 0.5
    return torch.cat([pos, neg]).float().mean().item()

def relation_accuracy(pred, A, B, truth_fn):
    """Acurácia de um predicado binário em todos os pares (a,b) contra a verdade geométrica."""
    out = pred(ltn.Variable("a", A), ltn.Variable("b", B)).value   # shape (len(A), len(B))
    return ((out > 0.5) == truth_fn(A, B)).float().mean().item()

x_left  = lambda A, B: A[:, 0].unsqueeze(1) < B[:, 0].unsqueeze(0)   # A está à esquerda de B
y_above = lambda A, B: A[:, 1].unsqueeze(1) > B[:, 1].unsqueeze(0)   # A está acima de B

def test_geometrical_model(num_per_shape=20):
    print("\n=== TESTANDO EM NOVOS DADOS ALEATÓRIOS ===")
    tc, ts, tt = generate_random_shapes(num_per_shape=num_per_shape)   # novo sorteio
    print(f"Teste: {len(tc)} círculos, {len(ts)} quadrados, {len(tt)} triângulos (novos)")

    with torch.no_grad():
        # Satisfação global da KB no conjunto de teste
        print(f"\nSatisfação da KB no TESTE: {define_geo_axioms(tc, ts, tt).item():.4f}")

        c_test = ltn.Variable("c_test", tc)
        s_test = ltn.Variable("s_test", ts)
        t_test = ltn.Variable("t_test", tt)

        # Consultas quantificadas
        q1 = Forall([c_test, s_test], Implies(And(is_circle(c_test), is_square(s_test)), left_of(c_test, s_test)))
        print(f"Q1: Todo círculo está à esquerda de todo quadrado? Satisfação: {q1.value.item():.4f}")
        q2 = Forall([t_test, c_test], Implies(And(is_triangle(t_test), is_circle(c_test)), above(t_test, c_test)))
        print(f"Q2: Todo triângulo está acima de todo círculo? Satisfação: {q2.value.item():.4f}")
        q3 = Forall([s_test, c_test], Not(left_of(s_test, c_test)))
        print(f"Q3: Nenhum quadrado está à esquerda de um círculo? Satisfação: {q3.value.item():.4f}")

        # Consultas sobre indivíduos específicos
        c1, s1, t1 = ltn.Constant(tc[0]), ltn.Constant(ts[0]), ltn.Constant(tt[0])
        print(f"\nPosições: círculo={tc[0].tolist()}, quadrado={ts[0].tolist()}, triângulo={tt[0].tolist()}")
        print(f"Q4: O 1º círculo é círculo?                 {is_circle(c1).value.item():.4f}")
        print(f"Q5: O 1º círculo é quadrado?                {is_square(c1).value.item():.4f}")
        print(f"Q6: O 1º triângulo é triângulo?             {is_triangle(t1).value.item():.4f}")
        print(f"Q7: 1º círculo à esquerda do 1º quadrado?   {left_of(c1, s1).value.item():.4f}")
        print(f"Q8: 1º quadrado à esquerda do 1º círculo?   {left_of(s1, c1).value.item():.4f}")
        print(f"Q9: 1º triângulo acima do 1º círculo?       {above(t1, c1).value.item():.4f}")
        print(f"Q10: 1º círculo sobrepõe o 1º quadrado?     {overlaps(c1, s1).value.item():.4f}")

        # Acurácia (limiar 0.5) contra a verdade geométrica calculada a partir das coordenadas
        print("\n--- Acurácia no conjunto de TESTE (limiar 0.5) ---")
        print(f"is_circle   : {class_accuracy(is_circle,   tc, [ts, tt]):.2%}")
        print(f"is_square   : {class_accuracy(is_square,   ts, [tc, tt]):.2%}")
        print(f"is_triangle : {class_accuracy(is_triangle, tt, [tc, ts]):.2%}")
        print(f"left_of  (círculos x quadrados, ida e volta): "
              f"{(relation_accuracy(left_of, tc, ts, x_left) + relation_accuracy(left_of, ts, tc, x_left)) / 2:.2%}")
        print(f"above    (triângulos x círculos, ida e volta): "
              f"{(relation_accuracy(above, tt, tc, y_above) + relation_accuracy(above, tc, tt, y_above)) / 2:.2%}")
        # Teste de estresse: TODOS os pares, inclusive pares da mesma classe (nunca vistos no treino)
        allp = torch.cat([tc, ts, tt])
        print(f"left_of  (todos os pares, teste de estresse): {relation_accuracy(left_of, allp, allp, x_left):.2%}")
        print(f"above    (todos os pares, teste de estresse): {relation_accuracy(above,   allp, allp, y_above):.2%}")

In [ ]:
# 6. EXECUÇÃO: treino (dados fixos) -> teste (dados novos)
print(f"Satisfação INICIAL no treino (antes de treinar): "
      f"{define_geo_axioms(train_circles, train_squares, train_triangles).item():.4f}")
train_geometrical_model(train_circles, train_squares, train_triangles, epochs=500, lr=0.05)
print(f"\nSatisfação FINAL no treino: "
      f"{define_geo_axioms(train_circles, train_squares, train_triangles).item():.4f}")
test_geometrical_model(num_per_shape=20)